# Binary Bernoulli Bandit Problem
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary $r_{t} = \left\{0,1\right\}$. The probability of getting reward `1` is unknown and needs to be estimated. The goal is to maximize the expected reward by selecting the best action at each time step.

* _Difference_: Unlike a completely general stochastic bandit problem, the binary Bernoulli bandit problem assumes the _agent models how the world responds_ using a (deceptively) simple reward distribution, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). Thus, _the agent has a model of the world_ (which is so super cool!).
* _Binary_: The reward distribution is binary. However, this is not as limiting as it may first appear. The experiment represented by the action $a$ can be a complex statement or function that _evaluates_ to a boolean value. Thus, we can model many complex scenarios which value `true` or `false.`

The Bernoulli distribution is a discrete probability distribution that returns a value of `1` with probability $p$ and value `0` with probability $1-p$. The probability mass function of the Bernoulli distribution is given by:
$$
\begin{equation*}
\texttt{Bern}(r; p) = \begin{cases}
p & \text{if } r = 1,\\
1-p & \text{if } r = 0.
\end{cases}
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward and $p\in[0,1]$ is the probability of getting reward `r = 1`. The expected reward of $X\sim\texttt{Bern}(r;p)$ is given by: $\mathbb{E}[X] = p$ and the variance is given by: $\text{Var}[X] = p(1-p)$. 
* _Ready to get your mind blown_? Ok, so here is the _cool part_: the agent models the parameter $p$ using a _probability distribution_ (e.g., [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution)) and updates this distribution as it observes rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874).

Yeah. That's cool. But how do we solve this problem?
___

## $\epsilon$-Greedy Binary Bernoulli Bandit
The $\epsilon$-greedy algorithm is simple and effective for solving the binary Bernoulli bandit problem. 

The algorithm selects the _best action_ with probability $1-\epsilon$ and selects a random action with probability $\epsilon$. The pseudo-code for the $\epsilon$-greedy algorithm is given below [(with more detail version can be found here)](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-7/L7c/docs/BBBPcode.pdf):

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha_{a} = 1$ and $\beta_{a} = 1$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. _Initialize_: Roll a random number $p\in\left[0,1\right]$ and compute a threshold $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$.
2. _Exploration_: If $p\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute the action $a_{t}$ and receive a reward $r_{t} = \left\{0,1\right\}$ from the _adversary_ (nature). 
3. _Exploitation_: Else if $p>\epsilon_{t}$, choose action $a^{\star}_{t}$, the action with the _highest expected probability of success_ (still a greedy choice), using the agents model of the world. Execute the action $a^{\star}_{t}$ and recieve a reward $r^{\star}_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature). 
    - We generate the highest probability estimate of success by sampling from the [Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm: $\mathbf{p}\gets\left\{\text{Beta}(\alpha(a)+\mathbf{S}(a),\beta(a)+\mathbf{F}(a))\mid\forall{a}\in\mathcal{A}\right\}$ where $\mathbf{S}(a)$ and $\mathbf{F}(a)$ are the number of successes and failures for arm $a$. The highest probability action is: $a^{\star} = \text{argmax}_{a\in\mathcal{A}}\left\{\mathbf{p}(a)\right\}$.
4. Update the success $\mathbf{S}(a^{\star})$ and failure $\mathbf{F}(a^{\star})$ arrays for the chosen arm $a^{\star}_{t}$ using the reward $r^{\star}_{t}$:
$$
\begin{equation*}
S(a^{\star}_{t}) \gets S(a^{\star}_{t}) + r^{\star}_{t},\quad F(a^{\star}_{t}) \gets F(a^{\star}_{t}) + (1-r^{\star}_{t})
\end{equation*}
$$

Using a model of the world allows the agent to make _probabilistic_ decisions about which actions to take. This is the essence of the Bayesian approach to bandit problems. The agent has a model of likely reward distribution for _each_ action and uses this model to select the best action at each time step.

If we step back, some decisions depend upon context. For example, understanding where we are on the planet would be handy if we were predicting the weather. Predicting product demand might depend upon the season, or determining which drugs to prescribe would depend upon the indication. Thus, _context_ is essential.

___


## $\epsilon$-Greedy Binary Contextual Bandit
The $\epsilon$-greedy algorithm can be extended to the binary contextual bandit problem by incorporating the _context_ into the agent's decision-making process. 

* In our simple approach, we'll assume that the context $s_{t}$ is a binary vector of length $d$ (i.e., $s_{t}\in\left\{0,1\right\}^{d}$). The agent maintains a separate model of the world for each context $s\in\left\{0,1\right\}^{d}$ and updates these models as it observes rewards. The agent selects the _best action_ based on the context $s_{t}$ at each time step and its associated program.
* Thus, we modify the $\epsilon$-greedy algorithm to incorporate an observation of the _context_ $s_{t}$ which can itself be _correct_ or _incorrect_. For example, the _context_ is a function of the physical position of the agent in a room, and the agent can observe this position with some error. The agent must then learn to make decisions based on the _observed_ context.

___